# Automatic Speech Recognition (ASR)

## What We'll Build

In this notebook, we'll explore **automatic speech recognition (ASR)** - the task of converting spoken audio into written text. We'll build intuitions about:

- The fundamental challenges of speech recognition (variability, accents, noise)
- Classic ASR pipeline: features → acoustic model → language model → text
- **CTC (Connectionist Temporal Classification)** - solving the alignment problem
- Building a working ASR system from scratch using RNN + CTC
- Modern approaches: attention mechanisms and self-supervised learning
- Using pretrained models like Whisper for production-quality ASR

## Why This Matters

ASR powers voice assistants, transcription services, accessibility tools, and human-computer interaction. Understanding ASR teaches us about:

- Sequence-to-sequence problems with alignment uncertainty
- The evolution from HMM-GMM → RNN-CTC → attention → self-supervised pretraining
- How to evaluate sequential outputs (WER, CER)
- Practical tradeoffs between custom models and pretrained systems

By the end, you'll understand how modern ASR systems work and when to use each approach.

## 1. Introduction: The Speech Recognition Problem

### The Core Challenge

Speech recognition is fundamentally about mapping **continuous audio waveforms** to **discrete text sequences**. This is hard because:

1. **Variable speaking rates** - people speak at different speeds
2. **Coarticulation** - sounds blend together when we speak
3. **Speaker variability** - different accents, pitch, tone
4. **Background noise** - real-world audio is rarely clean
5. **Alignment problem** - we don't know which audio frames correspond to which characters

The **alignment problem** is especially tricky: a character like "s" might span 5 frames at one speed but 15 frames at another. We need models that can handle this uncertainty.

## 2. Setup

### Install Required Packages

We'll need:
- `librosa` for audio processing
- `datasets` for loading speech data
- `transformers` for pretrained models
- Standard PyTorch stack

In [ ]:
# Install packages (run once)
# !pip install librosa datasets transformers soundfile jiwer

### Import Libraries

We'll organize imports by category: standard library, third-party, and our shared library.

In [ ]:
# Standard library
import warnings
from typing import List, Tuple, Optional

# Third-party
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from jiwer import wer, cer

# Local library
from aiml_notebooks import set_seed, get_device, CharacterTokenizer

# Auto-reload for development
%load_ext autoreload
%autoreload 2

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

### Set Random Seed and Device

For reproducibility, we set a random seed. We also configure the device for training.

In [ ]:
set_seed(42)
# Use CPU because CTC loss is not supported on MPS
device = get_device(prefer_cpu=True)
print(f"Using device: {device}")
print("Note: Using CPU because CTC loss is not yet supported on MPS (Apple Silicon)")

## 3. Generate Synthetic Speech Data

### Creating Simple Audio for Learning

For educational purposes, we'll generate synthetic speech-like data. This lets us focus on ASR concepts without dataset complexity. In production, you'd use real datasets like:

- **LibriSpeech** - English audiobooks (via `datasets` library)
- **Common Voice** - Crowd-sourced multilingual speech
- **TIMIT** - Phonetically-labeled continuous speech

Our synthetic audio will simulate simple spoken words with varying frequencies and durations.

In [ ]:
# Generate synthetic speech-like data
def generate_synthetic_audio(text: str, sr: int = 16000, base_freq: float = 200.0) -> np.ndarray:
    """Generate synthetic audio for a text string.
    
    Each character gets a unique frequency and the word has variable duration.
    This simulates the ASR alignment problem.
    """
    duration_per_char = 0.1  # seconds
    duration = len(text) * duration_per_char
    t = np.linspace(0, duration, int(sr * duration))
    
    audio = np.zeros_like(t)
    
    # Each character contributes a sine wave
    char_duration = duration / len(text)
    for i, char in enumerate(text):
        # Frequency based on character (simple hash)
        freq = base_freq + (ord(char) % 10) * 50
        
        # Time window for this character
        start_idx = int(i * char_duration * sr)
        end_idx = int((i + 1) * char_duration * sr)
        
        # Generate sine wave for character
        char_t = t[start_idx:end_idx] - t[start_idx]
        char_audio = np.sin(2 * np.pi * freq * char_t)
        
        # Apply envelope
        envelope = np.exp(-3 * char_t / char_duration)
        audio[start_idx:end_idx] += char_audio * envelope
    
    # Add slight noise
    noise = np.random.randn(len(audio)) * 0.05
    audio = audio + noise
    
    # Normalize
    audio = audio / (np.abs(audio).max() + 1e-6)
    
    return audio

# Create a simple vocabulary
words = [
    "yes", "no", "up", "down", "left", "right",
    "on", "off", "stop", "go", "one", "two",
    "three", "four", "five", "six", "seven",
    "eight", "nine", "zero", "hello", "world"
]

# Generate dataset
sample_rate = 16000
dataset_size = 500
dataset = []

np.random.seed(42)
for i in range(dataset_size):
    word = np.random.choice(words)
    audio = generate_synthetic_audio(word, sr=sample_rate)
    
    dataset.append({
        'audio': {'array': audio, 'sampling_rate': sample_rate},
        'label': word,
        'label_idx': words.index(word)
    })

# Create label names (like HF datasets feature)
label_names = words

print(f"Generated {len(dataset)} synthetic audio samples")
print(f"Vocabulary size: {len(label_names)} words")
print(f"Sample words: {label_names[:10]}")
print(f"\nExample entry:")
example = dataset[0]
print(f"  Label: '{example['label']}'")
print(f"  Audio shape: {example['audio']['array'].shape}")
print(f"  Sample rate: {example['audio']['sampling_rate']} Hz")

### Visualize Synthetic Audio Waveform

Let's look at what our synthetic audio looks like. The **waveform** shows amplitude over time.

In [ ]:
# Get a sample audio
sample_audio = dataset[0]['audio']['array']
sample_rate = dataset[0]['audio']['sampling_rate']
label = dataset[0]['label']

# Plot waveform
plt.figure(figsize=(12, 4))
time = np.arange(len(sample_audio)) / sample_rate
plt.plot(time, sample_audio)
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.title(f'Synthetic Audio Waveform - Word: "{label}"')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Duration: {len(sample_audio) / sample_rate:.2f} seconds")
print(f"Number of samples: {len(sample_audio)}")
print(f"This synthetic audio simulates '{label}' with variable frequencies per character.")

## 4. Classic ASR Pipeline Overview

### The Traditional Approach

Before deep learning, ASR systems followed this pipeline:

1. **Feature Extraction** - Convert raw audio → spectral features (MFCC, mel spectrogram)
2. **Acoustic Model** - Map features → phonemes (speech sounds) using HMM-GMM
3. **Pronunciation Dictionary** - Map phonemes → words
4. **Language Model** - Predict likely word sequences
5. **Decoder** - Find best path through possibilities

Modern deep learning ASR simplifies this:
- **Features** → still important, but learned representations can help
- **Acoustic + Language Model** → combined in one neural network
- **Decoder** → handled by CTC or attention mechanisms

We'll focus on the neural approach.

## 5. Feature Extraction: Mel Spectrograms

### Why Not Use Raw Audio?

Raw audio waveforms are **time-domain** representations. But speech is better understood in the **frequency domain** - what frequencies are present over time.

**Mel spectrograms** are ideal because:
- Show frequency content over time (2D representation)
- Use mel scale, which matches human perception
- Reduce dimensionality compared to raw audio
- Capture phonetic information well

### Compute Mel Spectrogram

We'll extract mel spectrograms using librosa. The parameters:
- `n_mels`: number of mel frequency bins (typically 40-128)
- `hop_length`: stride between frames (smaller = more time resolution)
- `n_fft`: FFT window size

In [ ]:
def compute_mel_spectrogram(audio: np.ndarray, sr: int, n_mels: int = 80) -> np.ndarray:
    """Compute log mel spectrogram from audio."""
    mel_spec = librosa.feature.melspectrogram(
        y=audio,
        sr=sr,
        n_mels=n_mels,
        n_fft=400,
        hop_length=160
    )
    # Convert to log scale (dB)
    log_mel_spec = librosa.power_to_db(mel_spec, ref=np.max)
    return log_mel_spec

# Compute for our sample
mel_spec = compute_mel_spectrogram(sample_audio, sample_rate)
print(f"Mel spectrogram shape: {mel_spec.shape}")
print(f"  Frequency bins (n_mels): {mel_spec.shape[0]}")
print(f"  Time frames: {mel_spec.shape[1]}")

### Visualize Mel Spectrogram

The spectrogram shows **frequency (y-axis)** vs **time (x-axis)**, with **color intensity** representing energy.

In [ ]:
plt.figure(figsize=(12, 5))
librosa.display.specshow(
    mel_spec,
    sr=sample_rate,
    hop_length=160,
    x_axis='time',
    y_axis='mel'
)
plt.colorbar(format='%+2.0f dB')
plt.title(f'Mel Spectrogram - Label: {label}')
plt.tight_layout()
plt.show()

print("Brighter regions = higher energy at that frequency and time")
print("You can see the temporal structure of the speech!")

## 6. The Alignment Problem

### Why We Need CTC

Imagine trying to align audio frames to text characters:

```
Audio frames:  [f1][f2][f3][f4][f5][f6][f7][f8][f9][f10]...
Text:          "hello"
```

Questions:
- Which frames correspond to 'h'? Maybe f1-f2?
- What about 'e'? f3-f5?
- What if someone speaks faster? 'h' might only be f1!

**The problem**: We don't have frame-level labels, only the final text. We need to train without knowing the alignment.

**CTC (Connectionist Temporal Classification)** solves this by:
1. Introducing a **blank token** (represented as `-`)
2. Allowing repeated predictions
3. Collapsing repeats and removing blanks to get final text
4. Marginalizing over all possible alignments during training

### CTC Alignment Example

Let's see how CTC alignments work with a concrete example.

In [ ]:
def collapse_ctc_alignment(alignment: str, blank: str = '-') -> str:
    """Collapse CTC alignment to final text.
    
    Rules:
    1. Remove consecutive duplicates
    2. Remove blank tokens
    """
    # Remove consecutive duplicates
    collapsed = []
    prev = None
    for char in alignment:
        if char != prev:
            collapsed.append(char)
        prev = char
    
    # Remove blanks
    result = ''.join([c for c in collapsed if c != blank])
    return result

# Example alignments all producing "hello"
alignments = [
    "hhhee-llll-oo",      # Repeated characters
    "h-e-l-l-o",          # Blanks between
    "hhh--eee--lll--ll--ooo",  # Mix of both
    "---hel--lo---",      # Blanks at edges
]

print("CTC Alignment Examples:\n")
for align in alignments:
    result = collapse_ctc_alignment(align)
    print(f"  {align:30s} → {result}")

print("\nAll alignments decode to the same text!")
print("This is how CTC handles variable speech rates.")

### Why Blanks Are Essential

Without blanks, we couldn't represent repeated characters! Consider "hello" vs "helo":

In [ ]:
# Without blanks between repeated characters, we can't distinguish:
alignment1 = "hel-lo"  # Decodes to "hello" (two l's)
alignment2 = "he-lo"   # Decodes to "helo" (one l)

print("Demonstrating the need for blank tokens:\n")
print(f"  {alignment1} → {collapse_ctc_alignment(alignment1)}")
print(f"  {alignment2} → {collapse_ctc_alignment(alignment2)}")
print("\nThe blank token lets us distinguish repeated vs single characters!")

## 7. CTC Loss: Mathematics

### How CTC Training Works

At each time step $t$, the model outputs a probability distribution over:
- All characters in the vocabulary
- The blank token

For a sequence with $T$ frames and target text $y$, CTC:

1. **Enumerates all valid alignments** - all ways to map frames to characters
2. **Sums their probabilities** - uses dynamic programming (forward-backward algorithm)
3. **Maximizes total probability** - $P(y|x) = \sum_{\text{alignments } \pi} P(\pi|x)$

The loss is simply:
$$\mathcal{L}_{\text{CTC}} = -\log P(y|x)$$

PyTorch implements this efficiently via `nn.CTCLoss`.

### CTC Loss Toy Example

Let's see CTC loss in action with a simple example.

In [ ]:
# Create a simple vocabulary: a, b, c + blank (index 0)
vocab = {'-': 0, 'a': 1, 'b': 2, 'c': 3}
inv_vocab = {v: k for k, v in vocab.items()}
vocab_size = len(vocab)

# Simulate model output: 6 timesteps, predicting over 4 classes
T = 6  # time steps
N = 1  # batch size
C = vocab_size  # number of classes

# Random log probabilities
torch.manual_seed(42)
log_probs = torch.randn(T, N, C).log_softmax(dim=2)

# Target: "ab" (indices [1, 2])
targets = torch.tensor([[1, 2]])  # shape: (N, S) where S is target length
input_lengths = torch.tensor([T])  # length of input sequence
target_lengths = torch.tensor([2])  # length of target

# Compute CTC loss
ctc_loss = nn.CTCLoss(blank=0, zero_infinity=True)
loss = ctc_loss(log_probs, targets, input_lengths, target_lengths)

print(f"Log probabilities shape: {log_probs.shape} (T={T}, N={N}, C={C})")
print(f"Target: {targets.squeeze().tolist()} → '{inv_vocab[1]}{inv_vocab[2]}'")
print(f"CTC Loss: {loss.item():.4f}")
print("\nThe loss measures how well all possible alignments match the target.")

## 8. Prepare Dataset for ASR

### Convert Labels to Text

The Speech Commands dataset has integer labels. We need to convert these to text strings for character-level ASR.

In [ ]:
# Get label names (already defined, but show them)
print(f"Found {len(label_names)} unique labels:")
print(label_names)

### Create Character Vocabulary

We'll build a character-level tokenizer from all label texts. For CTC, we need a **blank token** which we'll add explicitly at index 0.

In [ ]:
# Create tokenizer from our vocabulary
# Use a blank token '<blank>' for CTC
tokenizer = CharacterTokenizer(label_names, special_token='<blank>')

vocab_size = tokenizer.vocab_size
blank_idx = 0  # Special token is always at index 0

print(f"Vocabulary size: {vocab_size}")
print(f"First 10 chars: {tokenizer.chars[:10]}")
print(f"Blank token: '{tokenizer.special_token}' at index {blank_idx}")

### Test Tokenization

Let's verify our tokenizer works correctly.

In [ ]:
# Test encoding/decoding
test_text = "hello"
encoded = tokenizer.encode(test_text)
decoded = tokenizer.decode(encoded)

print(f"Original: {test_text}")
print(f"Encoded:  {encoded}")
print(f"Decoded:  {decoded}")
print(f"\nRound-trip successful: {test_text == decoded}")

### Create PyTorch Dataset

We'll wrap the data in a PyTorch Dataset that:
1. Loads audio
2. Computes mel spectrogram
3. Encodes text label to character indices

In [ ]:
class SpeechDataset(Dataset):
    def __init__(self, data_list, tokenizer, n_mels=80):
        self.data = data_list
        self.tokenizer = tokenizer
        self.n_mels = n_mels
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        # Get audio
        audio = item['audio']['array']
        sr = item['audio']['sampling_rate']
        
        # Compute mel spectrogram
        mel_spec = compute_mel_spectrogram(audio, sr, self.n_mels)
        
        # Get text label
        text = item['label'].lower()
        
        # Encode text
        encoded_text = self.tokenizer.encode(text)
        
        return {
            'mel_spec': torch.FloatTensor(mel_spec),  # (n_mels, T)
            'text': text,
            'encoded_text': torch.LongTensor(encoded_text),
            'spec_length': mel_spec.shape[1],
            'text_length': len(encoded_text)
        }

# Create dataset
speech_dataset = SpeechDataset(dataset, tokenizer)
print(f"Created dataset with {len(speech_dataset)} examples")

# Test it
sample = speech_dataset[0]
print(f"\nSample batch:")
print(f"  Mel spec shape: {sample['mel_spec'].shape}")
print(f"  Text: '{sample['text']}'")
print(f"  Encoded: {sample['encoded_text'].tolist()}")
print(f"  Spec length: {sample['spec_length']}")
print(f"  Text length: {sample['text_length']}")

### Custom Collate Function

Audio sequences have **variable lengths**. We need to pad them to create batches. For CTC loss, we also need to track the original lengths.

In [ ]:
def collate_fn(batch):
    """Collate variable-length sequences."""
    # Find max lengths
    max_spec_len = max(item['spec_length'] for item in batch)
    max_text_len = max(item['text_length'] for item in batch)
    n_mels = batch[0]['mel_spec'].shape[0]
    batch_size = len(batch)
    
    # Initialize padded tensors
    mel_specs = torch.zeros(batch_size, n_mels, max_spec_len)
    texts = torch.zeros(batch_size, max_text_len, dtype=torch.long)
    spec_lengths = torch.zeros(batch_size, dtype=torch.long)
    text_lengths = torch.zeros(batch_size, dtype=torch.long)
    text_strings = []
    
    # Fill in data
    for i, item in enumerate(batch):
        spec = item['mel_spec']
        spec_len = item['spec_length']
        text = item['encoded_text']
        text_len = item['text_length']
        
        mel_specs[i, :, :spec_len] = spec
        texts[i, :text_len] = text
        spec_lengths[i] = spec_len
        text_lengths[i] = text_len
        text_strings.append(item['text'])
    
    return {
        'mel_specs': mel_specs,
        'texts': texts,
        'spec_lengths': spec_lengths,
        'text_lengths': text_lengths,
        'text_strings': text_strings
    }

# Test collate function
test_loader = DataLoader(speech_dataset, batch_size=4, collate_fn=collate_fn)
test_batch = next(iter(test_loader))

print("Collated batch:")
print(f"  Mel specs shape: {test_batch['mel_specs'].shape}")
print(f"  Texts shape: {test_batch['texts'].shape}")
print(f"  Spec lengths: {test_batch['spec_lengths'].tolist()}")
print(f"  Text lengths: {test_batch['text_lengths'].tolist()}")
print(f"  Text strings: {test_batch['text_strings']}")

## 9. Build ASR Model: RNN + CTC

### Model Architecture

Our ASR model will have three parts:

1. **Convolutional layers** - extract local patterns from mel spectrogram
2. **Bidirectional LSTM** - model temporal dependencies in both directions
3. **Linear projection** - map to vocabulary size (including blank)

The output will be a sequence of probability distributions over characters at each time step.

In [ ]:
class ASRCTC(nn.Module):
    def __init__(self, n_mels: int, hidden_size: int, vocab_size: int, num_layers: int = 2):
        super().__init__()
        
        # Convolutional layers to reduce time dimension and extract features
        self.conv = nn.Sequential(
            nn.Conv1d(n_mels, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
        )
        
        # Bidirectional LSTM for sequence modeling
        self.lstm = nn.LSTM(
            input_size=128,
            hidden_size=hidden_size,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=True,
            dropout=0.1 if num_layers > 1 else 0
        )
        
        # Output projection to vocabulary
        self.fc = nn.Linear(hidden_size * 2, vocab_size)  # *2 for bidirectional
    
    def forward(self, mel_specs, spec_lengths):
        """
        Args:
            mel_specs: (batch, n_mels, time)
            spec_lengths: (batch,) actual lengths before padding
        Returns:
            log_probs: (time, batch, vocab_size) in log space
            output_lengths: (batch,) output sequence lengths
        """
        batch_size = mel_specs.size(0)
        
        # Conv expects (batch, channels, time)
        x = self.conv(mel_specs)  # (batch, 128, time)
        
        # Transpose for LSTM: (batch, time, features)
        x = x.transpose(1, 2)
        
        # Pack padded sequence for efficient LSTM processing
        x = nn.utils.rnn.pack_padded_sequence(
            x, spec_lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        
        # LSTM
        x, _ = self.lstm(x)
        
        # Unpack
        x, output_lengths = nn.utils.rnn.pad_packed_sequence(x, batch_first=True)
        
        # Project to vocabulary
        logits = self.fc(x)  # (batch, time, vocab_size)
        
        # CTC expects (time, batch, vocab_size) in log space
        log_probs = F.log_softmax(logits, dim=2)
        log_probs = log_probs.transpose(0, 1)  # (time, batch, vocab_size)
        
        return log_probs, output_lengths.to(mel_specs.device)

# Create model
model = ASRCTC(
    n_mels=80,
    hidden_size=128,
    vocab_size=vocab_size,
    num_layers=2
).to(device)

# Count parameters
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model created with {n_params:,} trainable parameters")
print(f"\nArchitecture:")
print(model)

### Test Forward Pass

Let's verify the model produces the right output shapes.

In [ ]:
# Test forward pass
model.eval()
with torch.no_grad():
    mel_specs = test_batch['mel_specs'].to(device)
    spec_lengths = test_batch['spec_lengths'].to(device)
    
    log_probs, output_lengths = model(mel_specs, spec_lengths)

print(f"Input shape: {mel_specs.shape} (batch, n_mels, time)")
print(f"Output shape: {log_probs.shape} (time, batch, vocab_size)")
print(f"Output lengths: {output_lengths.tolist()}")
print(f"\nLog probabilities sum to ~1 (in log space): {log_probs[0, 0].exp().sum().item():.4f}")

## 10. Training Setup

### Create Data Loaders

We'll split our data into train and validation sets.

In [ ]:
# Split dataset
train_size = int(0.8 * len(speech_dataset))
val_size = len(speech_dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(
    speech_dataset, [train_size, val_size]
)

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=collate_fn
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Train examples: {len(train_dataset)}")
print(f"Val examples: {len(val_dataset)}")

### Setup Loss and Optimizer

We'll use CTC loss and Adam optimizer.

In [ ]:
# Loss function
ctc_loss_fn = nn.CTCLoss(blank=blank_idx, zero_infinity=True)

# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Learning rate scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)

print("Training setup complete!")
print(f"  Loss: CTC (blank={blank_idx})")
print(f"  Optimizer: Adam (lr=0.001)")
print(f"  Scheduler: ReduceLROnPlateau")

## 11. CTC Decoding: Greedy Decoder

### From Model Outputs to Text

At inference time, we need to convert the model's probability distributions into actual text. The simplest approach is **greedy decoding**:

1. Take argmax at each timestep (most likely character)
2. Collapse consecutive duplicates
3. Remove blank tokens

This is fast but not optimal. **Beam search** (covered later) explores multiple paths for better results.

In [ ]:
def greedy_decode(log_probs, tokenizer, blank_idx=0):
    """
    Greedy CTC decoder.
    
    Args:
        log_probs: (time, batch, vocab_size)
        tokenizer: CharacterTokenizer
        blank_idx: index of blank token
    Returns:
        List of decoded strings
    """
    # Take argmax at each timestep
    _, max_indices = log_probs.max(dim=2)  # (time, batch)
    max_indices = max_indices.transpose(0, 1)  # (batch, time)
    
    decoded = []
    for sequence in max_indices:
        # Remove consecutive duplicates
        collapsed = []
        prev = None
        for idx in sequence.tolist():
            if idx != prev:
                collapsed.append(idx)
            prev = idx
        
        # Remove blanks
        filtered = [idx for idx in collapsed if idx != blank_idx]
        
        # Decode to text
        text = tokenizer.decode(filtered)
        decoded.append(text)
    
    return decoded

# Test greedy decoder
model.eval()
with torch.no_grad():
    mel_specs = test_batch['mel_specs'].to(device)
    spec_lengths = test_batch['spec_lengths'].to(device)
    
    log_probs, output_lengths = model(mel_specs, spec_lengths)
    predictions = greedy_decode(log_probs, tokenizer, blank_idx)

print("Greedy decoding results (untrained model):")
for i, (pred, target) in enumerate(zip(predictions, test_batch['text_strings'])):
    print(f"  [{i}] Predicted: '{pred:15s}' | Target: '{target}'")

print("\n(Random predictions - model is not trained yet!)")

## 12. Training Loop

### Define Training Step

Each training iteration:
1. Forward pass through model
2. Compute CTC loss
3. Backpropagate gradients
4. Update weights

In [ ]:
def train_epoch(model, loader, optimizer, ctc_loss_fn, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    n_batches = 0
    
    for batch in loader:
        # Move to device
        mel_specs = batch['mel_specs'].to(device)
        texts = batch['texts'].to(device)
        spec_lengths = batch['spec_lengths'].to(device)
        text_lengths = batch['text_lengths'].to(device)
        
        # Forward pass
        log_probs, output_lengths = model(mel_specs, spec_lengths)
        
        # CTC loss
        # Note: texts need to be concatenated for CTCLoss
        loss = ctc_loss_fn(
            log_probs,          # (T, N, C)
            texts,              # (N, S)
            output_lengths,     # (N,)
            text_lengths        # (N,)
        )
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        
        # Gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        total_loss += loss.item()
        n_batches += 1
    
    return total_loss / n_batches

print("Training function defined.")

### Define Validation Step

During validation, we:
1. Compute loss (no gradient computation)
2. Decode predictions
3. Calculate error rates (CER, WER)

In [ ]:
def validate_epoch(model, loader, ctc_loss_fn, tokenizer, blank_idx, device):
    """Validate for one epoch."""
    model.eval()
    total_loss = 0
    n_batches = 0
    all_predictions = []
    all_targets = []
    
    with torch.no_grad():
        for batch in loader:
            # Move to device
            mel_specs = batch['mel_specs'].to(device)
            texts = batch['texts'].to(device)
            spec_lengths = batch['spec_lengths'].to(device)
            text_lengths = batch['text_lengths'].to(device)
            
            # Forward pass
            log_probs, output_lengths = model(mel_specs, spec_lengths)
            
            # CTC loss
            loss = ctc_loss_fn(
                log_probs,
                texts,
                output_lengths,
                text_lengths
            )
            
            total_loss += loss.item()
            n_batches += 1
            
            # Decode predictions
            predictions = greedy_decode(log_probs, tokenizer, blank_idx)
            all_predictions.extend(predictions)
            all_targets.extend(batch['text_strings'])
    
    avg_loss = total_loss / n_batches
    
    # Calculate Character Error Rate (CER)
    cer_score = cer(all_targets, all_predictions)
    
    return avg_loss, cer_score, all_predictions, all_targets

print("Validation function defined.")

### Train the Model

Now let's train! We'll run for several epochs and track progress.

In [ ]:
# Training configuration
num_epochs = 10
best_cer = float('inf')

# History tracking
history = {
    'train_loss': [],
    'val_loss': [],
    'val_cer': []
}

print("Starting training...\n")

for epoch in range(num_epochs):
    # Train
    train_loss = train_epoch(model, train_loader, optimizer, ctc_loss_fn, device)
    
    # Validate
    val_loss, val_cer, predictions, targets = validate_epoch(
        model, val_loader, ctc_loss_fn, tokenizer, blank_idx, device
    )
    
    # Update scheduler
    scheduler.step(val_loss)
    
    # Track history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_cer'].append(val_cer)
    
    # Save best model
    if val_cer < best_cer:
        best_cer = val_cer
        best_epoch = epoch
    
    # Print progress
    print(f"Epoch {epoch+1}/{num_epochs}:")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Loss:   {val_loss:.4f}")
    print(f"  Val CER:    {val_cer:.4f}")
    
    # Show sample predictions every few epochs
    if (epoch + 1) % 3 == 0:
        print("  Sample predictions:")
        for i in range(min(3, len(predictions))):
            print(f"    Pred: '{predictions[i]:15s}' | Target: '{targets[i]}'")
    print()

print(f"Training complete!")
print(f"Best CER: {best_cer:.4f} at epoch {best_epoch+1}")

### Plot Training Curves

Let's visualize how the model learned over time.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss curves
axes[0].plot(history['train_loss'], label='Train', marker='o')
axes[0].plot(history['val_loss'], label='Validation', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('CTC Loss')
axes[0].set_title('Training Progress: Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# CER curve
axes[1].plot(history['val_cer'], label='Validation CER', marker='o', color='green')
axes[1].axhline(y=best_cer, color='r', linestyle='--', label=f'Best: {best_cer:.4f}')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Character Error Rate')
axes[1].set_title('Training Progress: CER')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Loss decreases → model learning alignments")
print("CER decreases → better character-level predictions")

## 13. Evaluation Metrics: CER and WER

### Understanding Error Rates

ASR systems are evaluated using:

1. **CER (Character Error Rate)** - Edit distance at character level
   - $\text{CER} = \frac{\text{insertions} + \text{deletions} + \text{substitutions}}{\text{total characters}}$
   
2. **WER (Word Error Rate)** - Edit distance at word level
   - $\text{WER} = \frac{\text{insertions} + \text{deletions} + \text{substitutions}}{\text{total words}}$

Lower is better. CER is more granular; WER is more commonly reported.

### Calculate Final Metrics

Let's get final CER and WER on the validation set.

In [ ]:
# Get predictions on full validation set
model.eval()
all_predictions = []
all_targets = []

with torch.no_grad():
    for batch in val_loader:
        mel_specs = batch['mel_specs'].to(device)
        spec_lengths = batch['spec_lengths'].to(device)
        
        log_probs, output_lengths = model(mel_specs, spec_lengths)
        predictions = greedy_decode(log_probs, tokenizer, blank_idx)
        
        all_predictions.extend(predictions)
        all_targets.extend(batch['text_strings'])

# Calculate metrics
cer_score = cer(all_targets, all_predictions)
wer_score = wer(all_targets, all_predictions)

print(f"Final Evaluation Metrics:")
print(f"  CER: {cer_score:.4f} ({cer_score*100:.2f}%)")
print(f"  WER: {wer_score:.4f} ({wer_score*100:.2f}%)")
print(f"\nInterpretation:")
print(f"  CER = {cer_score:.4f} means on average, {cer_score*100:.1f}% of characters are incorrect")
print(f"  WER = {wer_score:.4f} means on average, {wer_score*100:.1f}% of words are incorrect")

### Show Prediction Examples

Let's examine some specific predictions to understand where the model succeeds and fails.

In [ ]:
print("Sample Predictions:\n")
print(f"{'Prediction':<20s} {'Target':<20s} {'Correct?'}")
print("-" * 50)

correct = 0
for pred, target in zip(all_predictions[:20], all_targets[:20]):
    is_correct = pred == target
    if is_correct:
        correct += 1
    symbol = "✓" if is_correct else "✗"
    print(f"{pred:<20s} {target:<20s} {symbol}")

accuracy = correct / 20
print(f"\nExact match accuracy (first 20): {accuracy*100:.1f}%")

## 14. Beam Search Decoding

### Beyond Greedy Decoding

Greedy decoding takes the most likely character at each step. But this is **locally optimal**, not globally optimal.

**Beam search** explores multiple hypotheses in parallel:
1. Keep top-k most likely sequences (beam width = k)
2. At each step, extend all beams with all possible next characters
3. Keep only the k best extended sequences
4. Return the best final sequence

This often improves accuracy at the cost of more computation.

### Simple Beam Search Implementation

Here's a basic beam search decoder for CTC.

In [ ]:
def beam_search_decode(log_probs, tokenizer, blank_idx=0, beam_width=5):
    """
    Simplified beam search CTC decoder.
    
    Args:
        log_probs: (time, batch, vocab_size) - note: only supports batch_size=1
        tokenizer: CharacterTokenizer
        blank_idx: blank token index
        beam_width: number of beams to keep
    Returns:
        Decoded string
    """
    # Assume batch_size = 1 for simplicity
    log_probs = log_probs[:, 0, :]  # (time, vocab_size)
    
    # Initialize beam: list of (sequence, log_prob)
    # Sequence is list of indices (with blanks and repeats)
    beams = [([], 0.0)]  # Start with empty sequence
    
    # Process each timestep
    for t in range(log_probs.size(0)):
        new_beams = []
        
        # Expand each beam
        for sequence, score in beams:
            # Try all possible next characters
            for idx in range(log_probs.size(1)):
                new_seq = sequence + [idx]
                new_score = score + log_probs[t, idx].item()
                new_beams.append((new_seq, new_score))
        
        # Keep top beam_width
        new_beams.sort(key=lambda x: x[1], reverse=True)
        beams = new_beams[:beam_width]
    
    # Get best sequence
    best_sequence, best_score = beams[0]
    
    # Collapse CTC alignment
    collapsed = []
    prev = None
    for idx in best_sequence:
        if idx != prev:
            collapsed.append(idx)
        prev = idx
    
    # Remove blanks
    filtered = [idx for idx in collapsed if idx != blank_idx]
    
    # Decode
    text = tokenizer.decode(filtered)
    return text

print("Beam search decoder implemented.")
print("Note: This is a simplified version. Production systems use more sophisticated beam search.")

### Compare Greedy vs Beam Search

Let's compare the two decoding strategies on a few examples.

In [ ]:
# Get a few examples
model.eval()
test_batch = next(iter(val_loader))

with torch.no_grad():
    mel_specs = test_batch['mel_specs'].to(device)
    spec_lengths = test_batch['spec_lengths'].to(device)
    
    log_probs, output_lengths = model(mel_specs, spec_lengths)
    
    # Greedy decoding
    greedy_preds = greedy_decode(log_probs, tokenizer, blank_idx)
    
    # Beam search decoding (first example only, as it's slow)
    beam_pred = beam_search_decode(
        log_probs[:, :1, :],  # First example
        tokenizer,
        blank_idx,
        beam_width=5
    )

print("Decoding Strategy Comparison:\n")
print(f"Target:       '{test_batch['text_strings'][0]}'")
print(f"Greedy:       '{greedy_preds[0]}'")
print(f"Beam Search:  '{beam_pred}'")
print("\nBeam search may produce better results by exploring multiple paths.")

## 15. Attention-Based ASR: Listen, Attend, Spell

### Beyond CTC: Attention Mechanisms

While CTC solves the alignment problem, it has limitations:
- Makes conditional independence assumptions
- Can't model long-range dependencies well
- Doesn't use language model implicitly

**Attention-based models** (e.g., Listen, Attend, Spell) take a different approach:
1. **Encoder** - Process entire audio sequence → hidden states
2. **Attention** - Decoder learns to "attend" to relevant encoder states
3. **Decoder** - Autoregressively generate text character by character

Key advantage: The model learns alignment automatically through attention, and can use context from the entire utterance.

### Attention Mechanism Intuition

Attention asks: **"Which parts of the audio should I focus on to predict the next character?"**

At each decoding step:
1. Compute attention weights over all encoder states
2. Take weighted sum of encoder states (context vector)
3. Use context + previous prediction to generate next character

This is similar to seq2seq models with attention that we've seen before, but applied to audio → text.

In [ ]:
# Simple attention visualization
# Simulate attention weights for "hello" (5 characters, 20 time steps)
np.random.seed(42)

text = "hello"
T = 20  # time steps
S = len(text)  # sequence length

# Create attention matrix (each row = attention for one output character)
attention_matrix = np.zeros((S, T))
for i in range(S):
    # Attention peaks around position corresponding to character
    center = int(T * (i + 0.5) / S)
    attention_matrix[i, :] = np.exp(-0.5 * ((np.arange(T) - center) / 2) ** 2)
    attention_matrix[i, :] /= attention_matrix[i, :].sum()

# Plot
plt.figure(figsize=(12, 4))
plt.imshow(attention_matrix, aspect='auto', cmap='Blues', interpolation='nearest')
plt.xlabel('Audio Time Steps')
plt.ylabel('Output Characters')
plt.yticks(range(S), list(text))
plt.title('Attention Weights: Which audio frames to focus on for each character')
plt.colorbar(label='Attention Weight')
plt.tight_layout()
plt.show()

print("Each row shows attention distribution for generating one character.")
print("Brighter = more attention to that time step.")
print("Notice the diagonal pattern - the model learns to attend sequentially!")

## 16. Modern ASR: Self-Supervised Learning

### The Data Problem

Traditional ASR requires **labeled audio-text pairs**. This is expensive to collect at scale.

Modern approaches use **self-supervised pretraining**:
1. **Pretraining** - Learn representations from unlabeled audio
2. **Finetuning** - Adapt to transcription task with small labeled dataset

This is analogous to BERT for NLP.

### Wav2Vec 2.0 Overview

**Wav2Vec 2.0** (Facebook AI, 2020) popularized this approach:

**Pretraining objective**:
1. Mask spans of audio (like BERT masks tokens)
2. Learn to predict masked regions using contrastive loss
3. Use Transformer encoder to capture context

**Results**:
- SOTA performance with much less labeled data
- Can be finetuned for ASR with CTC or seq2seq head
- Works well for low-resource languages

**Key insight**: The model learns rich representations of speech sounds (phonemes, prosody) without explicit supervision.

### Whisper Architecture Overview

**Whisper** (OpenAI, 2022) takes a different approach:

**Training**:
- Trained on 680,000 hours of multilingual web data
- Uses encoder-decoder Transformer (like seq2seq)
- Multitask: transcription, translation, language detection, voice activity detection

**Architecture**:
```
Audio → Log-mel spec → Encoder (Transformer) → Decoder (Transformer) → Text
```

**Key features**:
- No CTC - uses attention-based decoder
- Handles multiple languages, accents, domains
- Strong zero-shot performance
- Various model sizes (tiny → large)

It's currently one of the most robust ASR systems available.

## 17. Using Pretrained Whisper

### Load Pretrained Model

Let's use the Hugging Face Transformers library to load a pretrained Whisper model.

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

# Load smallest Whisper model (for speed)
print("Loading Whisper model... (this may take a moment)")
processor = WhisperProcessor.from_pretrained("openai/whisper-tiny")
whisper_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny")
whisper_model = whisper_model.to(device)

print(f"Loaded Whisper-tiny model")
print(f"Model parameters: {sum(p.numel() for p in whisper_model.parameters()):,}")
print("\nThis is a production-quality model trained on 680k hours of data!")

### Transcribe Audio with Whisper

Whisper has a different preprocessing pipeline than our custom model. Let's transcribe some audio.

In [ ]:
# Get a sample from our dataset
sample = dataset[0]
audio = sample['audio']['array']
sr = sample['audio']['sampling_rate']
target_text = sample['label'].lower()

# Process audio for Whisper (expects 16kHz)
if sr != 16000:
    audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)

# Prepare input
inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
input_features = inputs.input_features.to(device)

# Generate transcription
with torch.no_grad():
    predicted_ids = whisper_model.generate(input_features)

# Decode
transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

print(f"Target:       '{target_text}'")
print(f"Whisper:      '{transcription.strip().lower()}'")
print(f"\nNote: Whisper is trained on real speech, so may not work well on synthetic audio.")
print(f"This demonstrates the interface - use real audio datasets for actual ASR!")

### Compare Custom Model vs Whisper

Let's compare our custom RNN-CTC model with pretrained Whisper on multiple examples.

In [ ]:
# Get a batch of examples
n_examples = 5
custom_correct = 0
whisper_correct = 0

print(f"{'Target':<15s} {'Custom Model':<15s} {'Whisper':<15s}")
print("-" * 50)

for i in range(n_examples):
    # Get sample
    sample = dataset[i]
    audio = sample['audio']['array']
    sr = sample['audio']['sampling_rate']
    target = sample['label'].lower()
    
    # Custom model prediction
    mel_spec = compute_mel_spectrogram(audio, sr)
    mel_spec_tensor = torch.FloatTensor(mel_spec).unsqueeze(0).to(device)
    spec_len = torch.LongTensor([mel_spec.shape[1]]).to(device)
    
    with torch.no_grad():
        log_probs, _ = model(mel_spec_tensor, spec_len)
        custom_pred = greedy_decode(log_probs, tokenizer, blank_idx)[0]
    
    # Whisper prediction
    if sr != 16000:
        audio_16k = librosa.resample(audio, orig_sr=sr, target_sr=16000)
    else:
        audio_16k = audio
    
    inputs = processor(audio_16k, sampling_rate=16000, return_tensors="pt")
    input_features = inputs.input_features.to(device)
    
    with torch.no_grad():
        predicted_ids = whisper_model.generate(input_features)
    whisper_pred = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0].strip().lower()
    
    # Track accuracy
    if custom_pred == target:
        custom_correct += 1
    if whisper_pred == target:
        whisper_correct += 1
    
    print(f"{target:<15s} {custom_pred:<15s} {whisper_pred:<15s}")

print("\nAccuracy:")
print(f"  Custom Model: {custom_correct}/{n_examples} ({100*custom_correct/n_examples:.1f}%)")
print(f"  Whisper:      {whisper_correct}/{n_examples} ({100*whisper_correct/n_examples:.1f}%)")
print("\nNote: Custom model trained on synthetic data shows the CTC mechanism.")
print("Whisper is designed for real speech - use real datasets for production ASR!")

## 18. Key Takeaways

### Evolution of ASR Systems

We've traced the evolution of automatic speech recognition:

**Classic Pipeline (pre-2010s)**:
- Hand-crafted features (MFCC, mel spectrograms)
- HMM-GMM acoustic models
- Separate language models
- Complex, multi-stage pipelines

**Deep Learning Era (2010s)**:
- RNN/LSTM + CTC loss
- End-to-end training
- Solves alignment problem elegantly
- Still uses hand-crafted features

**Attention Era (mid-2010s)**:
- Listen, Attend, Spell
- Encoder-decoder with attention
- Better long-range dependencies
- Implicit language modeling

**Self-Supervised Era (2020s)**:
- Wav2Vec 2.0, Whisper
- Pretrain on unlabeled audio
- Transformer architectures
- SOTA performance with less labeled data

### When to Use Each Approach

**Custom RNN-CTC Models**:
- Educational purposes
- Very constrained vocabulary (commands, digits)
- Need for interpretability
- Limited computational resources
- Specialized domains with lots of training data

**Attention-Based Models**:
- Complex sequences with long-range dependencies
- When you need better language modeling
- Have sufficient training data
- Can afford autoregressive decoding cost

**Pretrained Models (Whisper, Wav2Vec)**:
- **Almost always the best choice for production**
- Limited labeled training data
- Need multilingual support
- Want robust generalization
- Don't mind larger model size
- Can finetune for specific domains

### Core Concepts Mastered

1. **The Alignment Problem**: Speech varies in duration; we need models that handle this uncertainty

2. **CTC Loss**: Solves alignment by marginalizing over all possible alignments using blank tokens and collapse rules

3. **Feature Extraction**: Mel spectrograms convert audio to a representation that captures phonetic information

4. **Evaluation Metrics**: CER and WER measure edit distance at character and word levels

5. **Decoding Strategies**: Greedy (fast) vs beam search (better quality) tradeoff

6. **Attention Mechanisms**: Let the model learn which audio frames to focus on for each output

7. **Self-Supervised Pretraining**: Learn from unlabeled audio, then finetune - the modern standard

### Final Thought

ASR has gone from a specialized, fragile technology to robust, multilingual systems accessible via APIs. Understanding the fundamentals - CTC, attention, and self-supervised learning - helps you:
- Debug issues with production systems
- Finetune models for specific domains
- Know when custom solutions are worth building
- Appreciate the engineering that makes voice assistants work